# Multilabel Chest X-Ray Pathology Classification — V2 (Optimised)
## GCS Public Dataset · 7-Class · GPU-Accelerated · 3-Model Ensemble

---

**Runtime:** Google Colab — use **GPU T4** (Runtime → Change runtime type → T4 GPU)  
**Dataset:** `gs://gcs-public-data--healthcare-nih-chest-xray` (public NIH bucket — no manual download)  
**Classes (7):** Cardiomegaly · Effusion · Edema · Pneumothorax · Consolidation · Atelectasis · No Finding  
**Data volume:** ~40% of NIH ChestX-ray14 (~28–35K training images, stratified per class)  
**Models:** DenseNet-121 · ResNet50V2 · EfficientNetV2S → WA + Stacking ensembles  

---

### Changes from V1
| Change | V1 | V2 |
|---|---|---|
| Data source | Google Drive folder | GCS public bucket (direct) |
| Classes | 4 (incl. Infiltration) | 7 (Infiltration dropped; +Cardiomegaly/Edema/Pneumothorax/Consolidation) |
| Data size | 5,155 train | ~28–35K train (40% of NIH) |
| Device | CPU only | GPU (T4) with CPU fallback |
| Precision | float32 | mixed_float16 on GPU |
| Backbones | 2 | 3 (+EfficientNetV2S) |
| Augmentation | conservative | +rotation +saturation jitter on GPU |
| Max epochs | 10 fine-tune | 25 fine-tune (early stop patience=7) |


## Section 1: Environment Setup

Auto-detects GPU and configures mixed precision. Falls back to CPU transparently — no code changes needed.

In [ ]:
import os, sys, gc, io, math, json, random, shutil, warnings, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks, regularizers
from tensorflow.keras.applications import DenseNet121, ResNet50V2, EfficientNetV2S

from sklearn.metrics import (
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, f1_score, precision_score,
    recall_score, multilabel_confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ── Hardware detection ────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
HAS_GPU = len(gpus) > 0

if HAS_GPU:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print(f'GPU detected: {gpus[0].name}  →  mixed_float16 enabled')
else:
    N_CPUS = os.cpu_count() or 2
    tf.config.threading.set_inter_op_parallelism_threads(N_CPUS)
    tf.config.threading.set_intra_op_parallelism_threads(N_CPUS)
    tf.keras.mixed_precision.set_global_policy('float32')
    print(f'No GPU — CPU mode ({N_CPUS} cores), float32')

print(f'TensorFlow : {tf.__version__}')
print(f'Python     : {sys.version.split()[0]}')

plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'axes.grid.axis': 'y', 'grid.alpha': 0.3,
    'font.size': 11, 'axes.titlesize': 13, 'legend.fontsize': 9,
    'figure.facecolor': 'white',
})

## Section 2: Configuration

**Class selection — why these 7?**

Infiltration from V1 is **dropped**: it has the lowest published AUC (~0.73) and high label noise (it's a radiologically vague, catch-all term). The 4 new classes all have published AUC > 0.78.

| Class | Wang 2017 AUC | Why included |
|---|---|---|
| Cardiomegaly | 0.871 | Highest AUC in NIH; clear cardiac silhouette enlargement |
| Effusion | 0.811 | Distinctive pleural opacity |
| Edema | 0.805 | Bilateral perihilar opacity |
| Pneumothorax | 0.799 | Visible pleural line |
| Consolidation | 0.790 | Lobar opacity pattern |
| Atelectasis | 0.716 | Kept from V1 — volume loss |
| No Finding | 0.833 | Normal class |

**Epoch budget:** V2 sets `EPOCHS_FINETUNE = 25` with `patience = 7`. If val AUC stops improving for 7 consecutive epochs, training stops automatically. Extra epochs only consume time if they produce genuine improvement.

In [ ]:
class CFG:
    # ── GCS & local paths ─────────────────────────────────────────────────────
    GCS_BUCKET   = 'gs://gcs-public-data--healthcare-nih-chest-xray'
    WORK_DIR     = Path('/content/xray_v2')
    IMAGE_DIR    = WORK_DIR / 'images'
    TFRECORD_DIR = WORK_DIR / 'tfrecords'
    CKPT_DIR     = WORK_DIR / 'checkpoints'
    OUTPUT_DIR   = WORK_DIR / 'outputs'
    LOG_DIR      = WORK_DIR / 'logs'

    # ── Optional: persist outputs to Drive ───────────────────────────────────
    SAVE_TO_DRIVE   = True          # set False if Drive not mounted
    GDRIVE_SAVE_DIR = Path('/content/drive/MyDrive/xray_v2_results')

    # ── Classes (7, ordered by descending published AUC) ─────────────────────
    CLASSES = [
        'Cardiomegaly',    # ~0.87
        'Effusion',        # ~0.86  (Note: original paper said ~0.81; CheXNet pushed it higher)
        'Edema',           # ~0.81
        'Pneumothorax',    # ~0.80
        'Consolidation',   # ~0.79
        'Atelectasis',     # ~0.74  (kept from V1)
        'No Finding',      # ~0.83
    ]
    NUM_CLASSES = len(CLASSES)

    # ── Sampling caps (primary-label basis, training split only) ─────────────
    # Cardiomegaly and Edema are rare; we take all available (~2–3K each).
    # Rarer classes benefit from all examples; common ones are capped for balance.
    SAMPLE_CAPS = {
        'Cardiomegaly' : 9999,   # take all (~2.7K)
        'Effusion'     : 6000,
        'Edema'        : 9999,   # take all (~2.3K)
        'Pneumothorax' : 5000,
        'Consolidation': 4500,
        'Atelectasis'  : 6000,
        'No Finding'   : 12000,
    }

    # ── Image ─────────────────────────────────────────────────────────────────
    IMAGE_SIZE = 224
    CHANNELS   = 3

    # ── Training (GPU-adaptive) ───────────────────────────────────────────────
    BATCH_SIZE      = 32 if HAS_GPU else 16
    EPOCHS_WARMUP   = 5
    EPOCHS_FINETUNE = 25   # upper limit; early stopping cuts short if val AUC plateaus
    EARLY_STOP_PAT  = 7    # no improvement for 7 epochs → stop
    REDUCE_LR_PAT   = 4    # no improvement for 4 epochs → halve LR

    LR_WARMUP  = 1e-3
    LR_FINETUNE = 1e-4
    LR_MIN      = 1e-6

    WEIGHT_DECAY = 1e-4
    DROPOUT      = 0.35 if HAS_GPU else 0.40

    # ── Asymmetric Loss ───────────────────────────────────────────────────────
    ASL_GAMMA_POS = 1
    ASL_GAMMA_NEG = 4
    ASL_MARGIN    = 0.05

    # ── Data pipeline ─────────────────────────────────────────────────────────
    SHARDS_TRAIN   = 16 if HAS_GPU else 8
    SHARDS_VAL     = 4
    SHARDS_TEST    = 4
    SHUFFLE_BUFFER = 2048 if HAS_GPU else 512

    # ── Backbones ─────────────────────────────────────────────────────────────
    BACKBONES      = ['densenet121', 'resnet50v2', 'efficientnetv2s']
    BACKBONE_NAMES = ['DenseNet-121', 'ResNet50V2', 'EfficientNetV2S']

    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD  = [0.229, 0.224, 0.225]


for d in [CFG.WORK_DIR, CFG.IMAGE_DIR, CFG.TFRECORD_DIR,
          CFG.CKPT_DIR, CFG.OUTPUT_DIR, CFG.LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PALETTE = {
    'Cardiomegaly' : '#E63946', 'Effusion'     : '#F18F01',
    'Edema'        : '#A8DADC', 'Pneumothorax' : '#457B9D',
    'Consolidation': '#6A0572', 'Atelectasis'  : '#2E86AB',
    'No Finding'   : '#44BBA4',
}
CLS_COLORS   = [PALETTE[c] for c in CFG.CLASSES]
MODEL_COLORS = ['#264653', '#E76F51', '#2A9D8F', '#E9C46A', '#A8DADC']

print('Configuration loaded.')
print(f'Device      : {"GPU" if HAS_GPU else "CPU"}')
print(f'Batch size  : {CFG.BATCH_SIZE}')
print(f'Classes ({CFG.NUM_CLASSES})  : {CFG.CLASSES}')
print(f'Max epochs  : warm-up={CFG.EPOCHS_WARMUP}, fine-tune={CFG.EPOCHS_FINETUNE}')
print(f'Early stop  : patience={CFG.EARLY_STOP_PAT} epochs')

## Section 3: GCS Data Acquisition

The NIH ChestX-ray14 dataset is hosted in a **publicly readable** GCS bucket — no credentials needed.

**Steps:**
1. Download metadata CSV + official train/test split lists (< 5 MB, instant)
2. Parse labels, apply per-class sampling caps, 80/20 train/val split
3. Batch-download selected PNG images to `/content/xray_v2/images/` via `gsutil -m cp` (parallel, typically 5–15 min for ~30–40K images on Colab's fast network)

The download is **idempotent** — already-downloaded files are skipped on rerun.

In [ ]:
print('Downloading metadata from GCS (small files — instant)...')
META_CSV   = '/content/Data_Entry_2017.csv'
TRAIN_LIST = '/content/train_val_list.txt'
TEST_LIST  = '/content/test_list.txt'

!gsutil -q cp {CFG.GCS_BUCKET}/Data_Entry_2017.csv {META_CSV}
!gsutil -q cp {CFG.GCS_BUCKET}/train_val_list.txt  {TRAIN_LIST}
!gsutil -q cp {CFG.GCS_BUCKET}/test_list.txt        {TEST_LIST}

for f in [META_CSV, TRAIN_LIST, TEST_LIST]:
    kb = os.path.getsize(f) / 1024
    print(f'  {f.split("/")[-1]:30s} {kb:>8.0f} KB')
print('Metadata ready.')

In [ ]:
# ── Parse NIH metadata ────────────────────────────────────────────────────────
meta = pd.read_csv(META_CSV)
print(f'Total images in NIH CSV : {len(meta):,}')

meta['_label_set'] = meta['Finding Labels'].str.split('|').apply(set)

# Build binary columns for our 7 classes
for cls in CFG.CLASSES:
    meta[cls] = meta['_label_set'].apply(lambda s: int(cls in s))

# Primary label: first matching class in priority order (for stratified sampling)
def primary_label(label_set):
    for cls in CFG.CLASSES:          # ordered by AUC (Cardiomegaly first)
        if cls in label_set:
            return cls
    return None                       # image has only out-of-scope labels (e.g. Hernia only)

meta['primary'] = meta['_label_set'].apply(primary_label)
meta_filtered   = meta[meta['primary'].notna()].copy()

print(f'Images with >=1 target class : {len(meta_filtered):,}')
print(f'Images excluded (only out-of-scope labels) : {len(meta) - len(meta_filtered):,}')
print()
print('Primary-label distribution:')
print(meta_filtered['primary'].value_counts().to_string())

# ── Official splits ────────────────────────────────────────────────────────────
train_val_set = set(open(TRAIN_LIST).read().split())
test_set      = set(open(TEST_LIST).read().split())

meta_tv   = meta_filtered[meta_filtered['Image Index'].isin(train_val_set)]
meta_test = meta_filtered[meta_filtered['Image Index'].isin(test_set)]
print(f'\nTrain/val pool : {len(meta_tv):,}   Test pool : {len(meta_test):,}')

# ── Stratified sampling from train/val pool ───────────────────────────────────
parts = []
for cls in CFG.CLASSES:
    subset = meta_tv[meta_tv['primary'] == cls]
    cap    = CFG.SAMPLE_CAPS.get(cls, 5000)
    if len(subset) > cap:
        subset = subset.sample(cap, random_state=SEED)
    parts.append(subset)
    print(f'  {cls:15s}: {len(subset):,} sampled  (pool={len(meta_tv[meta_tv["primary"]==cls]):,}, cap={cap})')

tv_sampled = pd.concat(parts).drop_duplicates(subset='Image Index')
print(f'\nUnique train/val images after sampling : {len(tv_sampled):,}')

# 80/20 train/val split stratified by primary label
train_rows, val_rows = train_test_split(
    tv_sampled, test_size=0.20, random_state=SEED, stratify=tv_sampled['primary']
)
train_df = train_rows.reset_index(drop=True)
val_df   = val_rows.reset_index(drop=True)
test_df  = meta_test.reset_index(drop=True)   # no down-sampling on test

# Label counts
for split_name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    df['n_labels'] = df[CFG.CLASSES].sum(axis=1)
    print(f'\n{split_name} ({len(df):,} images) — mean labels/image: {df["n_labels"].mean():.2f}')
    for cls in CFG.CLASSES:
        print(f'  {cls:15s}: {df[cls].sum():,}')

In [ ]:
# ── Batch-download selected images from GCS ───────────────────────────────────
all_filenames = pd.concat([
    train_df['Image Index'], val_df['Image Index'], test_df['Image Index']
]).drop_duplicates().tolist()

already    = {p.name for p in CFG.IMAGE_DIR.glob('*.png')}
to_download = [f for f in all_filenames if f not in already]

print(f'Total unique images needed : {len(all_filenames):,}')
print(f'Already on disk             : {len(already):,}')
print(f'To download                 : {len(to_download):,}')

if to_download:
    list_file = '/content/gcs_download_list.txt'
    with open(list_file, 'w') as fh:
        for fname in to_download:
            fh.write(f'{CFG.GCS_BUCKET}/images/images/{fname}\n')
    print('\nStarting parallel download (gsutil -m cp)...')
    print('Expected time: 5-15 min depending on Colab network speed.')
    t0 = time.time()
    !gsutil -m -q cp -I {CFG.IMAGE_DIR} < {list_file}
    elapsed = time.time() - t0
    n_downloaded = len(list(CFG.IMAGE_DIR.glob('*.png'))) - len(already)
    print(f'Download complete: {n_downloaded:,} new files in {elapsed/60:.1f} min')
else:
    print('All images already on disk.')

# ── Add filepath column and verify ────────────────────────────────────────────
def add_and_verify(df, name):
    df = df.copy()
    df['filepath'] = df['Image Index'].apply(lambda f: str(CFG.IMAGE_DIR / f))
    before = len(df)
    df = df[df['filepath'].apply(os.path.exists)].reset_index(drop=True)
    n_miss = before - len(df)
    if n_miss > 0:
        print(f'  [{name}] {n_miss} files missing — dropped')
    return df

train_df = add_and_verify(train_df, 'train')
val_df   = add_and_verify(val_df,   'val')
test_df  = add_and_verify(test_df,  'test')

print(f'\nVerified splits: train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}')
print(f'Total unique images on disk: {len(list(CFG.IMAGE_DIR.glob("*.png"))):,}')

## Section 4: Exploratory Data Analysis

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Class Distribution — V2 (7 Classes, ~40% NIH)', fontsize=13,
             fontweight='semibold', y=1.02)

for ax, (df, title) in zip(axes, [
    (train_df, f'Train  (n={len(train_df):,})'),
    (val_df,   f'Val  (n={len(val_df):,})'),
    (test_df,  f'Test  (n={len(test_df):,})'),
]):
    counts = df[CFG.CLASSES].sum()
    bars   = ax.bar(CFG.CLASSES, counts.values, color=CLS_COLORS,
                    edgecolor='white', linewidth=1.2, width=0.65)
    ax.set_title(title); ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=35); ax.set_ylim(0, counts.max() * 1.22)
    for bar, (cls, cnt) in zip(bars, counts.items()):
        pct = 100 * cnt / len(df)
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + counts.max()*0.01,
                f'{cnt:,}\n({pct:.0f}%)', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig(CFG.OUTPUT_DIR / 'eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Imbalance ratios (needed for loss function reference)
pos_arr = train_df[CFG.CLASSES].sum(0).values.astype(np.float32)
neg_arr = len(train_df) - pos_arr
POS_WEIGHTS = (neg_arr / (pos_arr + 1e-8)).tolist()

print('Imbalance ratios (neg/pos) in training set:')
for cls, w in zip(CFG.CLASSES, POS_WEIGHTS):
    print(f'  {cls:15s}: {w:.2f}x')

## Section 5: TFRecord Conversion

Same binary serialisation as V1. Larger dataset → more shards (16 for train on GPU). Conversion runs once; subsequent cells skip it automatically.

In [ ]:
def _bytes_feature(v):      return tf.train.Feature(bytes_list=tf.train.BytesList(value=[v]))
def _float_list_feature(v): return tf.train.Feature(float_list=tf.train.FloatList(value=v))

def make_tfexample(filepath, label_array):
    img = Image.open(filepath).convert('RGB').resize(
        (CFG.IMAGE_SIZE, CFG.IMAGE_SIZE), Image.BILINEAR)
    buf = io.BytesIO(); img.save(buf, format='PNG')
    return tf.train.Example(features=tf.train.Features(feature={
        'image/encoded': _bytes_feature(buf.getvalue()),
        'label'        : _float_list_feature(label_array.tolist()),
    }))

def write_tfrecords(df, split_name, n_shards):
    shard_dir = CFG.TFRECORD_DIR / split_name
    shard_dir.mkdir(parents=True, exist_ok=True)
    existing = sorted(shard_dir.glob('*.tfrec'))
    if len(existing) == n_shards:
        mb = sum(p.stat().st_size for p in existing) / 1e6
        print(f'  [{split_name}] {n_shards} shards already exist ({mb:.0f} MB) — skip')
        return [str(p) for p in existing]
    for p in existing: p.unlink()
    groups = np.array_split(np.arange(len(df)), n_shards)
    paths  = []
    for si, idx_group in enumerate(tqdm(groups, desc=f'  {split_name}')):
        path = shard_dir / f'{split_name}-{si:04d}-of-{n_shards:04d}.tfrec'
        paths.append(str(path))
        with tf.io.TFRecordWriter(str(path)) as w:
            for i in idx_group:
                row = df.iloc[i]
                try:
                    ex = make_tfexample(
                        row['filepath'],
                        row[CFG.CLASSES].values.astype(np.float32))
                    w.write(ex.SerializeToString())
                except Exception as e:
                    print(f'\n  Skip {Path(row["filepath"]).name}: {e}')
    mb = sum(Path(p).stat().st_size for p in paths) / 1e6
    print(f'  [{split_name}] {n_shards} shards, {mb:.0f} MB')
    return paths

print('Converting to TFRecords (runs once)...')
train_tfr = write_tfrecords(train_df, 'train', CFG.SHARDS_TRAIN)
val_tfr   = write_tfrecords(val_df,   'val',   CFG.SHARDS_VAL)
test_tfr  = write_tfrecords(test_df,  'test',  CFG.SHARDS_TEST)
print('Done.')

## Section 6: tf.data Input Pipeline

GPU augmentation adds **random rotation (±8°)** and **saturation jitter** on top of V1's conservative set. These operations are negligible overhead on GPU but too slow on CPU — the code selects the right augmentation automatically.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
IM_MEAN  = tf.constant(CFG.IMAGENET_MEAN, dtype=tf.float32)
IM_STD   = tf.constant(CFG.IMAGENET_STD,  dtype=tf.float32)

FEAT_DESC = {
    'image/encoded': tf.io.FixedLenFeature([], tf.string),
    'label'        : tf.io.FixedLenFeature([CFG.NUM_CLASSES], tf.float32),
}

def parse_record(proto):
    feat  = tf.io.parse_single_example(proto, FEAT_DESC)
    image = tf.cast(tf.io.decode_png(feat['image/encoded'], channels=3), tf.float32) / 255.0
    return image, feat['label']

def augment_cpu(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.12)
    image = tf.image.random_contrast(image, lower=0.88, upper=1.12)
    sz    = tf.random.uniform([], int(CFG.IMAGE_SIZE*0.88), CFG.IMAGE_SIZE, dtype=tf.int32)
    image = tf.image.random_crop(image, [sz, sz, 3])
    image = tf.image.resize(image, [CFG.IMAGE_SIZE, CFG.IMAGE_SIZE])
    image = tf.clip_by_value(image + tf.random.normal(tf.shape(image), stddev=0.008), 0.0, 1.0)
    return image, label

def augment_gpu(image, label):
    """Stronger set: adds saturation jitter and random rotation."""
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.15)
    image = tf.image.random_contrast(image, lower=0.85, upper=1.15)
    image = tf.image.random_saturation(image, lower=0.85, upper=1.15)
    sz    = tf.random.uniform([], int(CFG.IMAGE_SIZE*0.85), CFG.IMAGE_SIZE, dtype=tf.int32)
    image = tf.image.random_crop(image, [sz, sz, 3])
    image = tf.image.resize(image, [CFG.IMAGE_SIZE, CFG.IMAGE_SIZE])
    # Random rotation ±8 degrees using tf.keras.layers.RandomRotation (stateless)
    angle = tf.random.uniform([], -8.0, 8.0)   # degrees
    image = tf.keras.layers.RandomRotation(
        factor=8/360, fill_mode='reflect', interpolation='bilinear'
    )(image[tf.newaxis], training=True)[0]
    image = tf.clip_by_value(image + tf.random.normal(tf.shape(image), stddev=0.006), 0.0, 1.0)
    return image, label

augment_fn = augment_gpu if HAS_GPU else augment_cpu

def normalize(image, label):
    return (tf.cast(image, tf.float32) - IM_MEAN) / IM_STD, label

def build_dataset(tfrec_paths, split='train', batch_size=CFG.BATCH_SIZE):
    is_train = (split == 'train')
    ds = tf.data.Dataset.from_tensor_slices(tfrec_paths)
    if is_train:
        ds = ds.shuffle(len(tfrec_paths), seed=SEED)
    ds = ds.interleave(
        lambda x: tf.data.TFRecordDataset(x, buffer_size=32*1024*1024),
        cycle_length=min(4, len(tfrec_paths)), block_length=8,
        num_parallel_calls=AUTOTUNE, deterministic=not is_train,
    )
    if is_train:
        ds = ds.shuffle(CFG.SHUFFLE_BUFFER, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(parse_record, num_parallel_calls=AUTOTUNE)
    if is_train:
        ds = ds.map(augment_fn, num_parallel_calls=AUTOTUNE)
    ds = ds.map(normalize, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=False).prefetch(AUTOTUNE)
    return ds

train_ds = build_dataset(train_tfr, 'train')
val_ds   = build_dataset(val_tfr,   'val')
test_ds  = build_dataset(test_tfr,  'test')

train_steps = math.ceil(len(train_df) / CFG.BATCH_SIZE)
val_steps   = math.ceil(len(val_df)   / CFG.BATCH_SIZE)
test_steps  = math.ceil(len(test_df)  / CFG.BATCH_SIZE)

print(f'Datasets built  |  augmentation: {"GPU (rotation+saturation)" if HAS_GPU else "CPU (conservative)"}')
print(f'  Train  : {len(train_df):,} images, {train_steps} batches')
print(f'  Val    : {len(val_df):,} images,  {val_steps} batches')
print(f'  Test   : {len(test_df):,} images,  {test_steps} batches')

## Section 7: Loss Function and Metrics

Asymmetric Loss (ASL) — unchanged from V1. See report Chapter 4.3 for derivation.

In [ ]:
class AsymmetricLoss(tf.keras.losses.Loss):
    """
    Ridnik et al. (2021) — Asymmetric Loss for multi-label classification.
    gamma_pos: focusing exponent for positive examples
    gamma_neg: focusing exponent for negative examples (gamma_neg > gamma_pos)
    margin   : probability shift that clips easy negatives to zero gradient
    """
    def __init__(self, gamma_pos=CFG.ASL_GAMMA_POS, gamma_neg=CFG.ASL_GAMMA_NEG,
                 margin=CFG.ASL_MARGIN, **kwargs):
        super().__init__(**kwargs)
        self.gp = gamma_pos; self.gn = gamma_neg; self.m = margin

    def call(self, y_true, y_pred):
        y_pred = tf.cast(y_pred, tf.float32)
        y_true = tf.cast(y_true, tf.float32)
        p      = tf.sigmoid(y_pred)
        pm     = tf.maximum(p - self.m, 0.0)
        lp = -y_true       * tf.pow(1.0 - p,  self.gp) * tf.math.log(p       + 1e-8)
        ln = -(1.0 - y_true) * tf.pow(pm, self.gn)      * tf.math.log(1.0 - pm + 1e-8)
        return tf.reduce_mean(lp + ln)

    def get_config(self):
        return dict(gamma_pos=self.gp, gamma_neg=self.gn,
                    margin=self.m, **super().get_config())


class MacroAUC(tf.keras.metrics.Metric):
    """Per-epoch macro-averaged AUC-ROC tracked by Keras."""
    def __init__(self, num_classes, name='macro_auc', **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self._aucs = [tf.keras.metrics.AUC(curve='ROC', name=f'auc_{i}')
                      for i in range(num_classes)]

    def update_state(self, y_true, y_pred, sample_weight=None):
        probs = tf.sigmoid(tf.cast(y_pred, tf.float32))
        for i, auc in enumerate(self._aucs):
            auc.update_state(y_true[:, i], probs[:, i], sample_weight)

    def result(self):
        return tf.reduce_mean([a.result() for a in self._aucs])

    def reset_state(self):
        for a in self._aucs: a.reset_state()

print(f'ASL ready: gamma_pos={CFG.ASL_GAMMA_POS}  gamma_neg={CFG.ASL_GAMMA_NEG}  margin={CFG.ASL_MARGIN}')

## Section 8: Model Architecture

**EfficientNetV2S (new in V2):** 21 M params, ImageNet top-1 = 83.9%. Compound scaling (depth + width + resolution) makes it the most accurate backbone here. On larger datasets (V2 vs V1) its advantage over DenseNet-121 is more pronounced.

All three backbones share an identical classification head.

In [ ]:
def build_backbone(name):
    kw = dict(include_top=False, weights='imagenet',
              input_shape=(CFG.IMAGE_SIZE, CFG.IMAGE_SIZE, CFG.CHANNELS))
    if name == 'densenet121':    return DenseNet121(**kw),    1024
    if name == 'resnet50v2':     return ResNet50V2(**kw),     2048
    if name == 'efficientnetv2s': return EfficientNetV2S(**kw), 1280
    raise ValueError(f'Unknown backbone: {name}')


def build_model(backbone_name):
    """Build full model: pretrained backbone + classification head."""
    base, _ = build_backbone(backbone_name)
    inp = keras.Input(shape=(CFG.IMAGE_SIZE, CFG.IMAGE_SIZE, CFG.CHANNELS))
    x   = base(inp, training=False)          # backbone frozen during warm-up
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(CFG.DROPOUT)(x)
    x   = layers.Dense(512, kernel_regularizer=regularizers.l2(CFG.WEIGHT_DECAY))(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation('gelu')(x)
    x   = layers.Dropout(CFG.DROPOUT * 0.5)(x)
    out = layers.Dense(CFG.NUM_CLASSES, dtype='float32', name='logits')(x)  # float32 for mixed prec safety
    return keras.Model(inp, out, name=backbone_name), base


print('Backbone summary:')
for bname, bdisp in zip(CFG.BACKBONES, CFG.BACKBONE_NAMES):
    m, _ = build_model(bname)
    print(f'  {bdisp:18s}: {m.count_params()/1e6:.1f} M params')
    del m; gc.collect()
print('All backbones OK.')

## Section 9: Training Engine

**Dynamic epoch budget:**  
- Phase 1 (warm-up, frozen backbone): 5 epochs — builds a stable head before unfreezing  
- Phase 2 (full fine-tune): up to 25 epochs, **early stop patience = 7**  
  → If val AUC doesn't improve for 7 consecutive epochs, training stops and best weights are restored  
  → Extra epochs beyond what's needed cost nothing  

Expected total training time (GPU T4, ~28K train images, batch=32):  
≈ 3–5 min/epoch → Phase 2 typically 12–18 epochs → ~45–90 min per backbone → ~3–5 hours total

In [ ]:
def make_callbacks(backbone_name):
    ckpt = str(CFG.CKPT_DIR / f'{backbone_name}_best.keras')
    cbs  = [
        callbacks.ModelCheckpoint(
            ckpt, monitor='val_macro_auc', mode='max',
            save_best_only=True, verbose=1),
        callbacks.EarlyStopping(
            monitor='val_macro_auc', mode='max',
            patience=CFG.EARLY_STOP_PAT,
            restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(
            monitor='val_macro_auc', mode='max',
            factor=0.5, patience=CFG.REDUCE_LR_PAT,
            min_lr=CFG.LR_MIN, verbose=1),
        callbacks.CSVLogger(str(CFG.LOG_DIR / f'{backbone_name}_log.csv'), append=False),
    ]
    return cbs, ckpt


def cosine_lr(ep, _, steps_per_ep, lr_max, lr_min, warmup_steps=0):
    step = ep * steps_per_ep
    if step < warmup_steps:
        return lr_min + (lr_max - lr_min) * step / max(warmup_steps, 1)
    t = (step - warmup_steps) / max(steps_per_ep * CFG.EPOCHS_FINETUNE - warmup_steps, 1)
    return lr_min + 0.5 * (lr_max - lr_min) * (1.0 + math.cos(math.pi * t))


def train_backbone(backbone_name, display_name):
    print(f'\n{"="*60}')
    print(f'  Training: {display_name}')
    print(f'{"="*60}')
    model, base = build_model(backbone_name)

    # ── Phase 1: Warm-up (frozen backbone) ──────────────────────────────────
    print(f'\n[Phase 1] Warm-up — frozen backbone, {CFG.EPOCHS_WARMUP} epochs')
    base.trainable = False
    lr_sched_wu = callbacks.LearningRateScheduler(
        lambda ep, lr: cosine_lr(ep, lr, train_steps, CFG.LR_WARMUP, CFG.LR_MIN)
    )
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=CFG.LR_WARMUP,
                                             weight_decay=CFG.WEIGHT_DECAY),
        loss=AsymmetricLoss(), metrics=[MacroAUC(CFG.NUM_CLASSES)]
    )
    wu_cbs, _ = make_callbacks(backbone_name)
    h1 = model.fit(train_ds, epochs=CFG.EPOCHS_WARMUP,
                   validation_data=val_ds,
                   callbacks=wu_cbs + [lr_sched_wu], verbose=1)
    best_wu = max(h1.history['val_macro_auc'])
    print(f'  Warm-up best val AUC: {best_wu:.4f}')

    # ── Phase 2: Full fine-tuning ────────────────────────────────────────────
    print(f'\n[Phase 2] Fine-tuning — all layers, up to {CFG.EPOCHS_FINETUNE} epochs')
    print(f'  Early stop patience: {CFG.EARLY_STOP_PAT} epochs')
    base.trainable = True
    wu2_steps = train_steps * 2          # 2-epoch linear warm-up within Phase 2
    lr_sched_ft = callbacks.LearningRateScheduler(
        lambda ep, lr: cosine_lr(ep, lr, train_steps, CFG.LR_FINETUNE,
                                  CFG.LR_MIN, wu2_steps)
    )
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=CFG.LR_FINETUNE,
                                             weight_decay=CFG.WEIGHT_DECAY),
        loss=AsymmetricLoss(), metrics=[MacroAUC(CFG.NUM_CLASSES)]
    )
    ft_cbs, ckpt = make_callbacks(backbone_name)
    h2 = model.fit(train_ds, epochs=CFG.EPOCHS_FINETUNE,
                   validation_data=val_ds,
                   callbacks=ft_cbs + [lr_sched_ft], verbose=1)

    ep_run  = len(h2.history['val_macro_auc'])
    best_ft = max(h2.history['val_macro_auc'])
    print(f'\n  Epochs run   : {ep_run} / {CFG.EPOCHS_FINETUNE}  ({"early stopped" if ep_run < CFG.EPOCHS_FINETUNE else "full budget"})')
    print(f'  Best val AUC : {best_ft:.4f}')
    print(f'  Checkpoint   : {ckpt}')
    return model, h2, ckpt

## Section 10: Train All Models

Each backbone is trained sequentially. `clear_session()` frees GPU memory between backbones.

> **Tip:** If training is interrupted (Colab disconnect), reload the notebook, re-run Sections 1-9 to rebuild the datasets, then load the checkpoint from disk with:  
> `model = tf.keras.models.load_model(CFG.CKPT_DIR / 'densenet121_best.keras', custom_objects={'AsymmetricLoss': AsymmetricLoss, 'MacroAUC': MacroAUC})`

In [ ]:
trained_models   = {}
training_hists   = {}
checkpoint_paths = {}

for bname, bdisp in zip(CFG.BACKBONES, CFG.BACKBONE_NAMES):
    gc.collect()
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)

    model, hist, ckpt = train_backbone(bname, bdisp)
    trained_models[bname]   = model
    training_hists[bname]   = hist
    checkpoint_paths[bname] = ckpt
    gc.collect()

print('\n' + '='*55)
print('  All backbones trained.')
print('='*55)
for bname, bdisp in zip(CFG.BACKBONES, CFG.BACKBONE_NAMES):
    h    = training_hists[bname]
    best = max(h.history['val_macro_auc'])
    ep   = len(h.history['val_macro_auc'])
    print(f'  {bdisp:18s}: val AUC = {best:.4f}  ({ep} epochs)')

In [ ]:
n = len(CFG.BACKBONES)
fig, axes = plt.subplots(1, n, figsize=(6*n, 5))
fig.suptitle('Validation Macro AUC — Training Progression', fontsize=13, fontweight='semibold')

for ax, bname, bdisp, col in zip(axes, CFG.BACKBONES, CFG.BACKBONE_NAMES, MODEL_COLORS[:n]):
    h  = training_hists[bname].history
    ep = range(1, len(h['val_macro_auc']) + 1)
    ax.plot(ep, h['val_macro_auc'], color=col, lw=2,   label='Val AUC')
    ax.plot(ep, h['macro_auc'],     color=col, lw=1.5, ls='--', alpha=0.6, label='Train AUC')
    best_ep = int(np.argmax(h['val_macro_auc'])) + 1
    best_v  = max(h['val_macro_auc'])
    ax.axvline(best_ep, color='#999', ls=':', lw=1)
    ax.scatter([best_ep], [best_v], s=80, color=col, zorder=5,
               label=f'Best: {best_v:.4f} (ep {best_ep})')
    ax.set_title(bdisp); ax.set_xlabel('Epoch'); ax.set_ylabel('Macro AUC')
    ax.set_ylim(0.5, 1.0); ax.legend()

plt.tight_layout()
plt.savefig(CFG.OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 11: Ensemble Construction

Three base models → same two strategies as V1, now over 3 backbones:

- **Weighted Average (WA):** 3000 Dirichlet samples over 3 weights, maximise val macro AUC  
- **Stacking:** 21-dimensional feature per image (3 models × 7 class probs), one logistic regression meta-learner per class trained on the validation set

In [ ]:
def get_probs(model, dataset, n_steps):
    logits = model.predict(dataset, steps=n_steps, verbose=1)
    return tf.sigmoid(tf.cast(logits, tf.float32)).numpy()

print('Collecting val predictions...')
val_probs  = {b: get_probs(trained_models[b], val_ds,  val_steps)  for b in CFG.BACKBONES}
print('\nCollecting test predictions...')
test_probs = {b: get_probs(trained_models[b], test_ds, test_steps) for b in CFG.BACKBONES}

val_labels  = val_df[CFG.CLASSES].values.astype(np.float32)
test_labels = test_df[CFG.CLASSES].values.astype(np.float32)

# Trim to exact lengths (last partial batch)
for d, ref in [(val_probs, len(val_labels)), (test_probs, len(test_labels))]:
    for k in d: d[k] = d[k][:ref]

print(f'\nPrediction arrays: val={val_labels.shape}  test={test_labels.shape}')

In [ ]:
# ── Weighted Average ─────────────────────────────────────────────────────────
rng = np.random.default_rng(SEED)
best_wa_auc, best_wa_w = 0.0, np.ones(len(CFG.BACKBONES)) / len(CFG.BACKBONES)

for _ in range(3000):
    w = rng.dirichlet(np.ones(len(CFG.BACKBONES)))
    p = sum(w[i] * val_probs[b] for i, b in enumerate(CFG.BACKBONES))
    try:
        auc = roc_auc_score(val_labels, p, average='macro')
        if auc > best_wa_auc: best_wa_auc, best_wa_w = auc, w
    except Exception: pass

print('Best WA weights:', {d: round(float(best_wa_w[i]), 3) for i, d in enumerate(CFG.BACKBONE_NAMES)})
print(f'Best WA val AUC: {best_wa_auc:.4f}')
test_probs['Ensemble-WA'] = sum(best_wa_w[i]*test_probs[b] for i, b in enumerate(CFG.BACKBONES))

# ── Stacking ─────────────────────────────────────────────────────────────────
val_feat  = np.hstack([val_probs[b]  for b in CFG.BACKBONES])
test_feat = np.hstack([test_probs[b] for b in CFG.BACKBONES])
scaler    = StandardScaler().fit(val_feat)
vf_s, tf_s = scaler.transform(val_feat), scaler.transform(test_feat)

stack_probs = np.zeros((len(test_df), CFG.NUM_CLASSES))
for ci, cls in enumerate(CFG.CLASSES):
    lr = LogisticRegression(C=1.0, max_iter=300, random_state=SEED)
    lr.fit(vf_s, val_labels[:, ci])
    stack_probs[:, ci] = lr.predict_proba(tf_s)[:, 1]

test_probs['Ensemble-Stack'] = stack_probs
print('\nEnsemble construction complete.')

In [ ]:
# ── Per-class threshold optimisation on val set ───────────────────────────────
val_wa = sum(best_wa_w[i]*val_probs[b] for i, b in enumerate(CFG.BACKBONES))

OPTIMAL_THRESHOLDS = {}
for ci, cls in enumerate(CFG.CLASSES):
    best_f1, best_t = 0.0, 0.5
    for t in np.arange(0.1, 0.91, 0.01):
        preds = (val_wa[:, ci] >= t).astype(int)
        f1    = f1_score(val_labels[:, ci], preds, zero_division=0)
        if f1 > best_f1: best_f1, best_t = f1, float(t)
    OPTIMAL_THRESHOLDS[cls] = best_t

print('Optimal thresholds (WA ensemble, val set):')
for cls, t in OPTIMAL_THRESHOLDS.items(): print(f'  {cls:15s}: {t:.2f}')

def evaluate(probs, labels, thresholds):
    r = {'macro_auc': roc_auc_score(labels, probs, average='macro'),
         'macro_ap' : average_precision_score(labels, probs, average='macro')}
    pc = {}
    for ci, cls in enumerate(CFG.CLASSES):
        t = thresholds.get(cls, 0.5); p = probs[:, ci]; y = labels[:, ci]
        preds = (p >= t).astype(int)
        pc[cls] = dict(auc=roc_auc_score(y, p), ap=average_precision_score(y, p),
                       f1=f1_score(y, preds, zero_division=0),
                       precision=precision_score(y, preds, zero_division=0),
                       recall=recall_score(y, preds, zero_division=0), threshold=t)
    r['macro_f1']   = np.mean([pc[c]['f1']        for c in CFG.CLASSES])
    r['macro_prec'] = np.mean([pc[c]['precision'] for c in CFG.CLASSES])
    r['macro_rec']  = np.mean([pc[c]['recall']    for c in CFG.CLASSES])
    r['per_class']  = pc
    return r

all_results = {}
model_keys  = CFG.BACKBONE_NAMES + ['Ensemble-WA', 'Ensemble-Stack']
all_test_probs = {d: test_probs[b] for b, d in zip(CFG.BACKBONES, CFG.BACKBONE_NAMES)}
all_test_probs.update({'Ensemble-WA': test_probs['Ensemble-WA'],
                       'Ensemble-Stack': test_probs['Ensemble-Stack']})

for k in model_keys:
    all_results[k] = evaluate(all_test_probs[k], test_labels, OPTIMAL_THRESHOLDS)

best_model_key = max(model_keys, key=lambda k: all_results[k]['macro_auc'])

print(f'\n{"Model":<22} {"AUC":>8} {"AP":>8} {"F1":>8}')
print('-'*44)
for k in model_keys:
    r = all_results[k]
    mark = '  <- best' if k == best_model_key else ''
    print(f'{k:<22} {r["macro_auc"]:>8.4f} {r["macro_ap"]:>8.4f} {r["macro_f1"]:>8.4f}{mark}')

## Section 12: Visualisations

In [ ]:
ls = ['-', '--', '-.', ':', (0,(3,1,1,1))]; bc = MODEL_COLORS[:len(model_keys)]
nc = min(4, CFG.NUM_CLASSES); nr = math.ceil(CFG.NUM_CLASSES / nc)
fig, axes = plt.subplots(nr, nc, figsize=(5*nc, 4.5*nr))
fig.suptitle('ROC Curves — All Models (Test Set)', fontsize=13, fontweight='semibold')
flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for ci, (cls, ax) in enumerate(zip(CFG.CLASSES, flat)):
    y = test_labels[:, ci]
    for ki, (k, col) in enumerate(zip(model_keys, bc)):
        p = all_test_probs[k][:, ci]
        fpr, tpr, _ = roc_curve(y, p); auc = roc_auc_score(y, p)
        ax.plot(fpr, tpr, color=col, lw=2.5 if 'Ensemble' in k else 1.5,
                ls=ls[ki%len(ls)], label=f'{k} ({auc:.3f})')
    ax.plot([0,1],[0,1],'k:',lw=0.8,alpha=0.4)
    ax.set_title(cls, fontweight='semibold', color=PALETTE[cls])
    ax.legend(fontsize=6.5, loc='lower right')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')

for ax in flat[CFG.NUM_CLASSES:]: ax.axis('off')
plt.tight_layout()
plt.savefig(CFG.OUTPUT_DIR / 'roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
mt = pd.DataFrame({
    cls: {m: all_results[best_model_key]['per_class'][cls][k]
          for m, k in [('AUC-ROC','auc'),('Avg Prec','ap'),('F1','f1'),
                       ('Precision','precision'),('Recall','recall')]}
    for cls in CFG.CLASSES
}).T

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(mt.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.025, label='Score')
ax.set_xticks(range(len(mt.columns))); ax.set_xticklabels(mt.columns)
ax.set_yticks(range(CFG.NUM_CLASSES)); ax.set_yticklabels(CFG.CLASSES)
ax.set_title(f'{best_model_key} — Per-Class Metrics (Test Set)', fontsize=12, fontweight='semibold')
for i in range(CFG.NUM_CLASSES):
    for j in range(len(mt.columns)):
        v = mt.values[i, j]
        ax.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=9,
                fontweight='semibold', color='white' if v < 0.35 else '#1a1a1a')
plt.tight_layout()
plt.savefig(CFG.OUTPUT_DIR / 'metrics_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(mt.round(4).to_string())

In [ ]:
bp = all_test_probs[best_model_key]
bpreds = np.stack([(bp[:, i] >= OPTIMAL_THRESHOLDS[cls]).astype(int)
                   for i, cls in enumerate(CFG.CLASSES)], axis=1)
mcm = multilabel_confusion_matrix(test_labels, bpreds)

nc = min(4, CFG.NUM_CLASSES); nr = math.ceil(CFG.NUM_CLASSES / nc)
fig, axes = plt.subplots(nr, nc, figsize=(4.5*nc, 4*nr))
fig.suptitle(f'Confusion Matrices — {best_model_key}', fontsize=12, fontweight='semibold')
flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for idx, (cls, col) in enumerate(zip(CFG.CLASSES, CLS_COLORS)):
    ax = flat[idx]; cm = mcm[idx]
    tn, fp, fn, tp_v = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred: No','Pred: Yes'],
                yticklabels=['True: No','True: Yes'],
                linewidths=0.8, linecolor='white',
                annot_kws={'size':12,'weight':'semibold'}, cbar=False)
    sens = tp_v/(tp_v+fn+1e-8); spec = tn/(tn+fp+1e-8)
    ax.set_title(f'{cls}\nSens: {sens:.2f}  Spec: {spec:.2f}',
                 fontsize=9, color=col, fontweight='semibold')

for ax in flat[CFG.NUM_CLASSES:]: ax.axis('off')
plt.tight_layout()
plt.savefig(CFG.OUTPUT_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 13: GradCAM Visual Explanations

Applied to DenseNet-121 (densest spatial feature maps → best spatial resolution for visualisation). One highest-scoring true positive per class.

In [ ]:
def find_last_conv(model):
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D): return layer.name
    raise ValueError('No Conv2D found')

def gradcam(model, img_t, class_idx, conv_name):
    gm = tf.keras.Model(inputs=model.inputs,
                        outputs=[model.get_layer(conv_name).output, model.output])
    with tf.GradientTape() as tape:
        inp = tf.expand_dims(img_t, 0)
        conv_out, preds = gm(inp, training=False)
        score = tf.cast(preds[0, class_idx], tf.float32)
    grads  = tf.cast(tape.gradient(score, conv_out), tf.float32)
    weights = tf.reduce_mean(grads, axis=[0,1,2])
    cam = tf.reduce_sum(conv_out[0] * weights, axis=-1).numpy()
    cam = np.maximum(cam, 0)
    if cam.max() > 0: cam /= cam.max()
    return cam

def colorize(cam, size):
    c = Image.fromarray(np.uint8(cam*255)).resize((size,size), Image.BILINEAR)
    return plt.get_cmap('inferno')(np.array(c, np.float32)/255)[:,:,:3]

dn   = trained_models['densenet121']
conv = find_last_conv(dn)
print(f'GradCAM layer: {conv}')

MN = np.array(CFG.IMAGENET_MEAN, np.float32)
MS = np.array(CFG.IMAGENET_STD,  np.float32)

def load_norm(fp):
    arr = np.array(Image.open(fp).convert('RGB').resize(
        (CFG.IMAGE_SIZE, CFG.IMAGE_SIZE), Image.BILINEAR), np.float32) / 255.0
    return (arr - MN) / MS

nc = 3; nr = CFG.NUM_CLASSES
fig, axes = plt.subplots(nr, nc, figsize=(11, 3.8*nr))
fig.suptitle('GradCAM Visual Explanations — DenseNet-121\nLeft: original  Centre: gradient  Right: overlay',
             fontsize=12, fontweight='semibold')

for ci, cls in enumerate(CFG.CLASSES):
    pos_idx = test_df.index[test_df[cls] == 1].tolist()
    if not pos_idx:
        for j in range(nc): axes[ci,j].axis('off')
        continue
    scores  = test_probs['densenet121'][:, ci]
    best_i  = pos_idx[int(np.argmax(scores[pos_idx]))]
    fp      = test_df.iloc[best_i]['filepath']
    p_score = float(scores[best_i])
    img_t   = load_norm(fp)
    cam     = gradcam(dn, tf.constant(img_t), ci, conv)
    hm      = colorize(cam, CFG.IMAGE_SIZE)
    orig    = np.array(Image.open(fp).convert('RGB').resize(
        (CFG.IMAGE_SIZE, CFG.IMAGE_SIZE), Image.BILINEAR), np.float32) / 255.0
    ov      = np.clip(0.55*orig + 0.45*hm, 0, 1)
    axes[ci,0].imshow(orig)
    axes[ci,0].set_ylabel(f'{cls}\np={p_score:.3f}', fontsize=9,
                           fontweight='semibold', color=PALETTE[cls])
    axes[ci,1].imshow(hm); axes[ci,2].imshow(ov)
    if ci == 0:
        for j, t in enumerate(['Original','Gradient map','Overlay']): axes[0,j].set_title(t)
    for j in range(nc): axes[ci,j].set_xticks([]); axes[ci,j].set_yticks([])

plt.tight_layout()
plt.savefig(CFG.OUTPUT_DIR / 'gradcam.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 14: Final Results Export

In [ ]:
# Save evaluation CSV
rows = []
for k in model_keys:
    for cls in CFG.CLASSES:
        d = all_results[k]['per_class'][cls]
        rows.append(dict(model=k, class_name=cls,
                         auc=d['auc'], avg_precision=d['ap'],
                         f1=d['f1'], precision=d['precision'],
                         recall=d['recall'], threshold=d['threshold']))
    rows.append(dict(model=k, class_name='MACRO',
                     auc=all_results[k]['macro_auc'],
                     avg_precision=all_results[k]['macro_ap'],
                     f1=all_results[k]['macro_f1'],
                     precision=all_results[k]['macro_prec'],
                     recall=all_results[k]['macro_rec'], threshold=None))

pd.DataFrame(rows).to_csv(CFG.OUTPUT_DIR / 'evaluation_results_v2.csv', index=False)

with open(CFG.OUTPUT_DIR / 'optimal_thresholds.json', 'w') as f:
    json.dump(OPTIMAL_THRESHOLDS, f, indent=2)

with open(CFG.OUTPUT_DIR / 'config_v2.json', 'w') as f:
    json.dump({'classes': CFG.CLASSES, 'num_classes': CFG.NUM_CLASSES,
               'batch_size': CFG.BATCH_SIZE, 'device': 'GPU' if HAS_GPU else 'CPU',
               'best_model': best_model_key,
               'macro_auc': all_results[best_model_key]['macro_auc']}, f, indent=2)

# Copy to Google Drive
if CFG.SAVE_TO_DRIVE:
    try:
        CFG.GDRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
        for fpath in sorted(CFG.OUTPUT_DIR.glob('*')):
            shutil.copy(fpath, CFG.GDRIVE_SAVE_DIR / fpath.name)
        print(f'Results saved to Drive: {CFG.GDRIVE_SAVE_DIR}')
    except Exception as e:
        print(f'Drive save skipped ({e}) — results are in {CFG.OUTPUT_DIR}')

# Create zip for download
shutil.make_archive('/content/xray_v2_results', 'zip', str(CFG.WORK_DIR))
print('Download zip: /content/xray_v2_results.zip  (Files panel → right-click → Download)')

# Final summary
print('\n' + '='*65)
print(f'  FINAL RESULTS — V2  |  {CFG.NUM_CLASSES} classes  |  {"GPU" if HAS_GPU else "CPU"}')
print(f'  Train: {len(train_df):,} images  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}')
print('-'*65)
print(f'  {"Model":<22} {"AUC":>8} {"AP":>8} {"F1":>8} {"Prec":>8} {"Rec":>8}')
print('-'*65)
for k in model_keys:
    r = all_results[k]; mark = '  <- best' if k == best_model_key else ''
    print(f'  {k:<22} {r["macro_auc"]:>8.4f} {r["macro_ap"]:>8.4f} '
          f'{r["macro_f1"]:>8.4f} {r["macro_prec"]:>8.4f} {r["macro_rec"]:>8.4f}{mark}')
print('='*65)
print('\nPer-class AUC — best model:')
for cls in CFG.CLASSES:
    v = all_results[best_model_key]['per_class'][cls]['auc']
    bar = '█' * int(v * 30)
    print(f'  {cls:15s}: {v:.4f}  {bar}')